# 13 — Reproducibility check

**Purpose.** Same seed gives bit-identical results; data checksums verify; software environment is recorded.

**Author:** Guy M. Kaptue T. · **Execution:** independent, seeded (`SEED = 0`), demonstration scale (CPU, about a minute). Outputs: `reports/figures/13_reproducibility_check_*.png|pdf` and `reports/notebooks/13_reproducibility_check.json`.

> **Data statement.** SYN-IDS-v1 is completely synthetic despite its name (scikit-learn `make_classification`); it is not real intrusion data and not patient data. The public healthcare datasets are de-identified research tables; no real hospital data is used, all federated clients are simulated locally in one process, and raw source files are not redistributed.

In [1]:
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
def _root():
    p = Path(os.environ.get("FL_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "pyproject.toml").exists() and (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set FL_ROOT")
ROOT = _root(); sys.path.insert(0, str(ROOT / "src")); os.environ["FL_ROOT"] = str(ROOT)
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
SEED = 0; np.random.seed(SEED)
NB_ID = "13_reproducibility_check"
FIG = ROOT / "reports" / "figures"; OUT = ROOT / "reports" / "notebooks"; FIG.mkdir(parents=True, exist_ok=True); OUT.mkdir(parents=True, exist_ok=True)
PAL = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#E69F00", "#56B4E9", "#000000"]   # Okabe-Ito, colour-blind safe
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.prop_cycle": matplotlib.cycler(color=PAL), "axes.grid": True, "grid.alpha": .25, "legend.frameon": False})
SUMMARY = {}
def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG / f"{NB_ID}_{name}.{ext}", bbox_inches="tight")
    plt.show()
def save_summary():
    (OUT / f"{NB_ID}.json").write_text(json.dumps(SUMMARY, indent=2, default=float)); print("summary ->", OUT / f"{NB_ID}.json")

## 1. Bit-identical reruns

In [2]:
from federated_learning.synthetic_data import make_dataset
from federated_learning.data_partitioning import partition
from federated_learning.servers.orchestrator import run_fl
from federated_learning.training import FLConfig
from federated_learning.utils.hashing import sha256_arrays
d = make_dataset(n_samples=3000, seed=SEED); parts = partition("dirichlet", d.y_train, 8, SEED, alpha=.5)
a = run_fl(FLConfig(rounds=5, seed=3), d, parts); b = run_fl(FLConfig(rounds=5, seed=3), d, parts); c = run_fl(FLConfig(rounds=5, seed=4), d, parts)
chk = dict(same_seed_identical=bool(np.array_equal(a.w, b.w)), different_seed_differs=bool(not np.array_equal(a.w, c.w)), sha256_same_seed=sha256_arrays(a.w) == sha256_arrays(b.w))
print(chk); assert all(chk.values()); SUMMARY["checks"] = chk
dist = {"same seed (3 vs 3)": float(np.linalg.norm(a.w - b.w)), "different seed (3 vs 4)": float(np.linalg.norm(a.w - c.w))}
fig, ax = plt.subplots(figsize=(4.2, 2.6)); ax.bar(list(dist), list(dist.values()), color=[PAL[2], PAL[1]]); ax.set_ylabel("L2 distance of final weights"); ax.set_title("Reruns are bit-identical")
save_fig(fig, "rerun_distance")

{'same_seed_identical': True, 'different_seed_differs': True, 'sha256_same_seed': True}


## 2. Dataset checksums and environment

In [3]:
from federated_learning.preprocessing import SPECS
from federated_learning.preprocessing.download import raw_path
from federated_learning.utils.hashing import sha256_file
rows = []
for k, s in SPECS.items():
    for f in s.files:
        p = raw_path(k, f.filename)
        if f.url and p.exists(): rows.append(dict(dataset=k, file=f.filename, ok=sha256_file(p) == f.sha256))
display(pd.DataFrame(rows)); assert all(r["ok"] for r in rows)
import importlib.metadata as md, platform
env = dict(python=platform.python_version(), platform=platform.platform(), **{p: md.version(p) for p in ("numpy", "scipy", "scikit-learn", "torch", "pandas", "matplotlib")}); print(json.dumps(env, indent=1)); SUMMARY["environment"] = env; save_summary()

,dataset,file,ok
0,pima_diabetes,pima.csv,True
1,cleveland_heart_disease,heart.csv,True
2,uci_heart_disease_multisite,cleveland14.csv,True
3,uci_heart_disease_multisite,hungarian14r.csv,True
4,uci_heart_disease_multisite,switzerland14.csv,True
5,uci_heart_disease_multisite,long_beach_va14.csv,True


{
 "python": "3.13.10",
 "platform": "macOS-26.5.1-arm64-arm-64bit-Mach-O",
 "numpy": "2.4.2",
 "scipy": "1.17.1",
 "scikit-learn": "1.9.0",
 "torch": "2.13.0",
 "pandas": "3.0.2",
 "matplotlib": "3.10.8"
}
summary -> /Users/guykaptue/Downloads/federated_learning_security/reports/notebooks/13_reproducibility_check.json


## Limitations
Demonstration scale (few rounds, one seed unless stated): numbers here illustrate the workflow and are not the study's reported results, which come from the full experiment pipeline in `experiments/results/` with multiple seeds and confidence intervals.